# 🚀 Nanbeige4.1-3B-q4_K_M 一键部署 & 聊天测试

| 项目 | 信息 |
|------|------|
| 模型 | [Nanbeige4.1-3B-q4_K_M](https://ollama.com/softw8/Nanbeige4.1-3B-q4_K_M) |
| 参数量 | 3.9B |
| 量化 | Q4_K_M (4-bit) |
| 模型大小 | ~2.5 GB |
| 运行时 | 建议使用 **GPU** (免费 T4 即可) |

### 使用步骤
1. **更改运行时类型** → 点击菜单 `代码执行程序` → `更改运行时类型` → 选择 `T4 GPU` → 保存
2. **按顺序运行**下方所有 Cell
3. 在最后的聊天 Cell 中开始对话

---

## Step 1: 安装 Ollama

In [ ]:
%%bash
# 1) 安装 zstd（Ollama 解压依赖）
apt-get update -qq && apt-get install -y -qq zstd > /dev/null 2>&1
echo "✅ zstd 已安装"

# 2) 安装 Ollama
curl -fsSL https://ollama.com/install.sh | sh

# 3) 用完整路径启动服务（%%bash 子 shell 不会继承新 PATH）
nohup /usr/local/bin/ollama serve > /tmp/ollama.log 2>&1 &
sleep 3

# 4) 验证
echo "✅ Ollama 版本: $(/usr/local/bin/ollama --version 2>&1)"
echo "✅ 服务状态: $(curl -s http://127.0.0.1:11434/api/tags > /dev/null && echo '运行中' || echo '未启动')"

## Step 2: 下载模型

> 模型约 2.5GB，下载时间取决于网络速度（通常 1-3 分钟）

In [ ]:
%%bash
MODEL="softw8/Nanbeige4.1-3B-q4_K_M"

echo "⏳ 正在下载模型 $MODEL (~2.5GB)..."
/usr/local/bin/ollama pull "$MODEL"

echo ""
echo "✅ 模型下载完成！"
/usr/local/bin/ollama list

## Step 3: 速度基准测试

测试模型的推理速度（首字延迟 + 生成速度）

In [ ]:
import json
import time
import urllib.request

MODEL = "softw8/Nanbeige4.1-3B-q4_K_M"
HOST = "http://127.0.0.1:11434"

test_prompts = [
    "你好，请用一句话介绍你自己。",
    "请用中文写一首关于春天的四行短诗。",
    "What is the capital of France?",
]

print("=" * 55)
print("  ⚡ 速度基准测试")
print("=" * 55)

results = []

for i, prompt in enumerate(test_prompts):
    label = prompt[:35] + "..." if len(prompt) > 35 else prompt
    print(f"\n  [{i+1}/{len(test_prompts)}] {label}")

    payload = json.dumps({
        "model": MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {"num_predict": 128}
    }).encode()

    req = urllib.request.Request(
        f"{HOST}/api/generate",
        data=payload,
        headers={"Content-Type": "application/json"}
    )

    t0 = time.time()
    with urllib.request.urlopen(req, timeout=120) as resp:
        data = json.loads(resp.read())
    total_time = time.time() - t0

    eval_count = data.get("eval_count", 0)
    eval_duration = data.get("eval_duration", 0) / 1e9
    prompt_eval_duration = data.get("prompt_eval_duration", 0) / 1e6
    response = data.get("response", "")

    tps = eval_count / eval_duration if eval_duration > 0 else 0
    results.append(tps)

    print(f"    首字延迟: {prompt_eval_duration:.0f} ms")
    print(f"    生成速度: {tps:.1f} tokens/s")
    print(f"    生成数量: {eval_count} tokens")
    print(f"    总耗时:   {total_time:.2f}s")
    print(f"    回复预览: {response[:80]}...")

print(f"\n{'=' * 55}")
print(f"  📊 平均生成速度: {sum(results)/len(results):.1f} tokens/s")
print(f"  📊 最快: {max(results):.1f} tok/s | 最慢: {min(results):.1f} tok/s")
print("=" * 55)

## Step 4: 💬 开始聊天

运行下方 Cell 后，在输入框中输入消息即可与模型对话。

- 输入 `quit` 或 `exit` 退出
- 输入 `clear` 清空对话历史
- 支持多轮上下文对话

In [ ]:
import json
import urllib.request

MODEL = "softw8/Nanbeige4.1-3B-q4_K_M"
HOST = "http://127.0.0.1:11434"

chat_history = []  # 多轮对话历史

def chat(user_message: str) -> str:
    """发送消息并获取模型回复（流式输出）。"""
    global chat_history

    chat_history.append({"role": "user", "content": user_message})

    payload = json.dumps({
        "model": MODEL,
        "messages": chat_history,
        "stream": True,
        "options": {
            "temperature": 0.7,
            "num_predict": 512,
        }
    }).encode()

    req = urllib.request.Request(
        f"{HOST}/api/chat",
        data=payload,
        headers={"Content-Type": "application/json"}
    )

    full_response = ""
    with urllib.request.urlopen(req, timeout=120) as resp:
        buf = ""
        while True:
            chunk = resp.read(1)
            if not chunk:
                break
            buf += chunk.decode("utf-8", errors="replace")
            while "\n" in buf:
                line, buf = buf.split("\n", 1)
                line = line.strip()
                if not line:
                    continue
                try:
                    event = json.loads(line)
                except json.JSONDecodeError:
                    continue
                content = event.get("message", {}).get("content", "")
                if content:
                    # 流式打印
                    print(content, end="", flush=True)
                    full_response = content
                if event.get("done"):
                    break

    chat_history.append({"role": "assistant", "content": full_response})
    print()  # 换行
    return full_response


# ==================== 聊天主循环 ====================
print("=" * 55)
print("  🤖 Nanbeige4.1-3B 聊天")
print("  输入消息开始对话 | quit 退出 | clear 清空历史")
print("=" * 55)
print()

while True:
    try:
        user_input = input("\n🧑 你: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\n\n👋 再见！")
        break

    if not user_input:
        continue
    if user_input.lower() in ("quit", "exit", "q"):
        print("\n👋 再见！")
        break
    if user_input.lower() == "clear":
        chat_history = []
        print("\n🗑️ 对话历史已清空")
        continue

    print("\n🤖 模型: ", end="", flush=True)
    chat(user_input)

---

### 🔧 高级用法

如果需要通过 API 调用（例如从另一个 Cell 中调用）：

In [ ]:
# 单次 API 调用示例
import json, urllib.request

payload = json.dumps({
    "model": "softw8/Nanbeige4.1-3B-q4_K_M",
    "messages": [
        {"role": "user", "content": "用三句话介绍量子计算"}
    ],
    "stream": False
}).encode()

resp = urllib.request.urlopen(
    urllib.request.Request(
        "http://127.0.0.1:11434/api/chat",
        data=payload,
        headers={"Content-Type": "application/json"}
    ),
    timeout=60
)

result = json.loads(resp.read())
print(result["message"]["content"])